In [ ]:
import logging
import pandas as pd

from zenml import step

class IngestData:
    def __init__(self,data: str):
        self.data = data
    
    def get_data(self) -> pd.DataFrame:
        return pd.read_cs(self.data)
        logging.info("data is succesfully loaded")
        
@step
def ingest_df(data:str) -> pd.DataFrame:
    try:
        logging.info("starting new raw data ingest")
        ingest_data = IngestData(data)
        df = ingest_data.get_data()
        return df
        logging.info("data is completed successfully ")
    except Exception as e:
        logging.error(f"while error in feaching")
    

In [ ]:
from logging
from typing import Union

from abc import ABC,abstractmethod

class DataStrategy(ABC):
    def handle_data(self,data : str) -> Union[pd.DataFrame,pd.Series]:
        pass
    
    
class PreProcessorStrategy(DataStrategy):
    def handle_data(self, data)->pd.DataFrame:
        try:
            data =  data.drop([
                "customer_id",
                "customer_name",
                "date",
                "time"
            ],axis=1)
            
            data["age"] = data["age"].fillna(data["age"].mean())
            
            return data
        
        except Exception as e :
            logging.error(f"error in preprocessor: {e}")

#============================
# data dividing 
#============================

class DataDividing(DataStrategy):
    def handle_data(self, data)->Union[pd.DataFrame,pd.Series]:
        try:
            x =  data.drop("target",axis=1)
            y = data["target"]
            
            x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42)
            
            return x_train,x_test,y_train,y_test 
        except Exception as e:
            logging.error(f"error in divide strategy: {e}")
            
#============================
# content
#============================

class DataCleaning(DataStrategy):
    def __init__(self,data: pd.DataFrame,strategy:DataStrategy):
        self.data = data
        self.strategy = strategy
        
    def handle_data(self, data):
        return self.strategy,self.handle_data(self.data)

In [ ]:
from logging
from pandas pd
from typing import tuple, Annotated
from from zenml import step
from src.data_clean import DataPreprocessStrategy,DataDivideStrategy,DataCleaning

@step
def clean_df(data:pd.DataFrame)->tuple[
    Annotated[pd.DataFrame,"x_train"],
    Annotated[pd.DataFrame,"x_test"],
    Annotated[pd.Series,"y_train"],
    Annotated[pd.Series,"y_test"]
]:
    try:
        processor_strategy = DataPreprocessStrategy()
        data_cleaning = DataCleaning(data,processor_strategy)
        processed_data = data_cleaning.handle_data()
        
        # dividing
        divide_strategy = DataDivideStrategy()
        data_devide = DataCleaning(processed_data,divide_strategy)
        x_train,x_test,y_train,y_test = data_devide.handle_data()
        
        return x_train,x_test,y_train,y_test
    except Exception as e:
        logging.error(f"error in clean_df: {e}")

In [ ]:
import logging
import pandas as pd

from typing import Tuple, Annotated

from zenml import step

from src.data_clean import (
    PreProcessStrategy,
    DivideData,
    DataCleaning
)


@step
def clean_df(
    data: pd.DataFrame
) -> Tuple[
    Annotated[pd.DataFrame, "x_train"],
    Annotated[pd.DataFrame, "x_test"],
    Annotated[pd.Series, "y_train"],
    Annotated[pd.Series, "y_test"]
]:

    try:

        # Preprocessing
        preprocess_strategy = PreProcessStrategy()

        data_cleaning = DataCleaning(
            data,
            preprocess_strategy
        )

        processed_data = data_cleaning.handle_data()

        logging.info("Data preprocessing completed")

        # Divide data
        divide_strategy = DivideData()

        data_divide = DataCleaning(
            processed_data,
            divide_strategy
        )

        x_train, x_test, y_train, y_test = (
            data_divide.handle_data()
        )

        logging.info("Data splitting completed")

        return (
            x_train,
            x_test,
            y_train,
            y_test
        )

    except Exception as e:

        logging.error(f"Error in clean_df step: {e}")

        

In [ ]:
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier

from abc import ABC,abstractmethod

class Model(ABC):
    
    @abstractmethod
    def train(self,x_train,y_train):
        pass
    
    @abstractmethod
    def optimization(self,trial,x_train,x_test,y_train,y_test):
        pass
    
    
class RandomForestModel(Model):
    def train(self,x_train,y_train, **kwargs):
        
        reg = RandomForestClassifier(**kwargs)
        reg.fit(x_train,y_train)
        return reg
    
    def optimization(self, trial, x_train, x_test, y_train, y_test):
        n_estimator = trial.suggest_int("n_estimator",1,200)
        max_depth = trial.suggest_int("max_depth",2,20)
        min_sample_split = trial.suggest_int("min_sample",2,15)
        
        reg = self.train(x_train,x_test,n_estimator=n_estimator,max_depth=max_depth,min_sample_split=min_sample_split)
        
        return reg.score(x_test,y_test)
    
    
class AdaBoostModel(Model):
    def train(self, x_train, y_train,**kwargs):
        
        reg = AdaBoostClassifier(**kwargs)
        reg.fit(x_train,y_train)
        return reg
    
    def optimization(self, trial, x_train, x_test, y_train, y_test):
        
        n_estimator = trial.suggest_int("n_estimator",1,200)
        max_depth = trial.suggest_int("max_depth",2,20)
        min_sample_split = trial.suggest_int("min_sample",2,15)
        
        reg = self.train(x_train,x_test,n_estimator=n_estimator,max_depth=max_depth,min_sample_split=min_sample_split)
        
        return reg.score(x_test,y_test)
    
    
class HyperparameterTuning:
    def __init__(self,model,x_train,x_test,y_train,y_test):
        self.model = model
        self.x_train = x_train
        self.x_test = x_test
        self.y_train = y_train
        self.y_test = y_test
        
        
       